In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [5]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [6]:
from sklearn.model_selection import train_test_split

# Adjust "text" and "label" to your actual column names
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(800,) (200,)


## Data Preprocessing

In [7]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [10]:
import re

def clean_html(text):
    # 1. Remove inline JavaScript and CSS (tags AND their contents)
    text = re.sub(r"<script\b[^>]*>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<style\b[^>]*>.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)

    # 2. Remove HTML comments (before tags, since comments can contain '>')
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # 3. Remove the remaining tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Tidy up: collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train = X_train.apply(clean_html)
X_test = X_test.apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [11]:
def clean_text(text):
    # Remove all special characters (keep only letters and whitespace)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove numbers (already gone after the line above, but explicit for clarity)
    text = re.sub(r"\d+", " ", text)

    # Remove prefixed 'b' (from bytes-literal artifacts like b'hello')
    text = re.sub(r"^b\s+", "", text)

    # Remove all single characters surrounded by spaces
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)

    # Remove single characters from the start
    text = re.sub(r"^[a-zA-Z]\s+", "", text)

    # Substitute multiple spaces with a single space
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to lowercase
    text = text.lower()

    return text

X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")  

stop_words = set(stopwords.words("english"))   # set = fast lookups

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in stop_words)

X_train = X_train.apply(remove_stopwords)
X_test = X_test.apply(remove_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Roberto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [14]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

nltk.download("punkt")
nltk.download("punkt_tab")                      
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng") 

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    """Map a Penn Treebank POS tag to a WordNet POS tag."""
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text(text):
    tokens = word_tokenize(text)                 # 1. break into words
    tagged = pos_tag(tokens)                     # 2. tag each word (noun, verb, ...)
    lemmas = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]  # 3. base form
    return " ".join(lemmas)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Roberto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Roberto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Roberto\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Roberto\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Build the bag-of-words matrix on the cleaned training text
cv = CountVectorizer()
bow = cv.fit_transform(X_train)
vocab = cv.get_feature_names_out()

def top_words(label, n=10):
    mask = (y_train == label).values              # rows belonging to this class
    counts = bow[mask].sum(axis=0).A1             # total count of each word in the class
    return (pd.Series(counts, index=vocab)
              .sort_values(ascending=False)
              .head(n))

top_ham = top_words(0)    # assuming 0 = ham
top_spam = top_words(1)   # assuming 1 = spam

print("Top 10 ham words:\n", top_ham, "\n")
print("Top 10 spam words:\n", top_spam)

Top 10 ham words:
 president    94
would        92
mr           85
percent      80
obama        80
state        77
pm           68
work         67
one          67
call         61
dtype: int64 

Top 10 spam words:
 money          761
account        634
us             600
bank           599
transaction    425
business       412
nbsp           387
fund           385
country        374
million        347
dtype: int64


## Extra features

In [20]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).

data_train = X_train.to_frame("preprocessed_text")
data_val = X_test.to_frame("preprocessed_text")

money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,money_mark,suspicious_words,text_len
442,dear good day hope fine cdear writting mail du...,1,1,1012
962,mr henry kaborethe chief auditor inchargeforei...,0,1,1964
971,,0,0,0
190,desk dr adamu ismalerauditing accounting manag...,1,1,387
551,dear friend name loi estrada wife mr josephest...,1,1,1489


## How would work the Bag of Words with Count Vectorizer concept?

In [21]:
cv = CountVectorizer()
X_train_bow = cv.fit_transform(data_train["preprocessed_text"])   # learn vocab on train only
X_val_bow = cv.transform(data_val["preprocessed_text"])           # reuse the same vocab

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load the vectorizer
tfidf = TfidfVectorizer()

# 2. Vectorize the dataset (fit on train only, reuse the vocabulary on validation)
X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

# 3. Print the shape
print(X_train_tfidf.shape)
print(X_val_tfidf.shape)

(800, 29099)
(200, 29099)


## And the Train a Classifier?

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

y_val = y_test

clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_val_tfidf)

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code